# R3b: recover Saved Output and complete frozen diagnostics
Use the existing **private** bingjiangli/minicells Notebook and Save & Run All. Version65 is only a discovery candidate. No training, Pilot, Confirmation or task configuration. Required existing Secrets: KAGGLE_API_TOKEN and GITHUB_TOKEN; HF_TOKEN is optional read-only recovery.
If a task-boundary pause occurs, the same Notebook can be saved and run again; it discovers validated prior Saved Output packages. This Notebook does not claim it can start its own future cloud session.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

ORCHESTRATOR_SHA = "575ed27f177c69653179b896bc4b5cda8f953a0e"
ENGINE_SHA = "ed2accbc90f541325ca2f6c53229e9c81475768d"
KAGGLE_CLI_SHA = "6894864ff4095cf98e9e36d3a694ff5fc9ba3818"
BASE = Path("/kaggle/temp/minicells-r3b-output-recovery")
CODE = BASE / ("orchestrator-" + ORCHESTRATOR_SHA[:12])
ENGINE = BASE / ("engine-" + ENGINE_SHA[:12])
OUTPUT = Path("/kaggle/working/minicells-r3b-posthoc")
RELATIVE = Path("research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b")
os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1")

def run(args, cwd=None):
    print("[Output Recovery] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)

secrets = UserSecretsClient()
for name in ("KAGGLE_API_TOKEN", "GITHUB_TOKEN"):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable existing Kaggle Secret " + name) from None
    if not os.environ[name]:
        raise RuntimeError("Missing existing Kaggle Secret " + name)
if not os.environ.get("HF_TOKEN"):
    try:
        os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN").strip()
    except Exception:
        print("Optional HF recovery unavailable; Kaggle input and local result persistence remain enabled.", flush=True)

patterns = ["/pyproject.toml", "/src/", "/shared/python/",
    "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
    "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
    "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
    "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
    "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
    "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/"]
for checkout, sha in ((CODE, ORCHESTRATOR_SHA), (ENGINE, ENGINE_SHA)):
    if sha.startswith("__"):
        raise RuntimeError("Notebook implementation has not been pinned")
    if not (checkout / ".git").exists():
        checkout.mkdir(parents=True, exist_ok=False)
        run(["git", "init", "--quiet"], checkout)
        run(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], checkout)
        run(["git", "sparse-checkout", "init", "--no-cone"], checkout)
        run(["git", "sparse-checkout", "set", "--no-cone", *patterns], checkout)
        run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", sha], checkout)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], checkout)
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip() != sha:
        raise RuntimeError("SOURCE_SHA_MISMATCH")
# Versioned Output support and its v<N> label fix are pinned to the audited official source.
run([sys.executable, "-m", "pip", "install", "--quiet", "kaggle @ git+https://github.com/Kaggle/kaggle-cli.git@" + KAGGLE_CLI_SHA])
for module, requirement in (("tokenizers", "tokenizers"), ("matplotlib", "matplotlib")):
    if importlib.util.find_spec(module) is None:
        run([sys.executable, "-m", "pip", "install", "--quiet", requirement])
run(["kaggle", "--version"])
print({"ACTION": "full-analysis-from-kaggle-output", "MEASUREMENT_ENGINE": ENGINE_SHA,
       "EXECUTION_ORCHESTRATOR": ORCHESTRATOR_SHA, "SOURCE_CANDIDATE_VERSION": 65,
       "HF_INPUT_REQUIRED": False, "TRAINING_UPDATES": 0, "CONFIRMATION": "SEALED"}, flush=True)
run([sys.executable, CODE / RELATIVE / "output_analysis.py", "--engine-root", ENGINE,
     "--output", OUTPUT, "--cache", BASE / "historical-output-cache", "--device", "cuda:0",
     "--session-seconds", "6900", "--publish-github"], CODE)
print("OUTPUT_VERSION_PENDING_SAVE: this Save & Run All must retain Output. A later launch verifies saved results and resumes automatically.", flush=True)
